# Transpacific Passenger Simulation — How the Data Is Generated

This notebook walks through the simulation architecture layer by layer, showing the math, the config, and the output at each stage.

---

## The three-layer model

```
┌─────────────────────────────────────────────┐
│  Layer 1 — Market demand  (demand.py)        │
│  λ_OD(t) = μ_OD · seasonal · DOW · trend    │
│             · shock · noise                  │
│  μ_OD is a FIXED constant, NOT from seats   │
└──────────────────┬──────────────────────────┘
                   │  expected_demand (float)
                   ▼
┌─────────────────────────────────────────────┐
│  Layer 2 — Supply allocation  (engine.py)   │
│  flight_share = seats_f / Σ seats_OD        │
│  flight_expected = λ_OD · flight_share      │
│  Supply acts here — demand already exists   │
└──────────────────┬──────────────────────────┘
                   │  flight_expected per flight
                   ▼
┌─────────────────────────────────────────────┐
│  Layer 3 — Booking / censoring  (sim.py)    │
│  Poisson arrivals per cabin per DTP         │
│  FCFS: accept until cabin is full           │
│  oracle_pax vs observed_pax + BOH curve     │
└─────────────────────────────────────────────┘
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'#0e1117','axes.facecolor':'#161b22','axes.edgecolor':'#30363d',
    'axes.labelcolor':'#c9d1d9','axes.titlecolor':'#f0f6fc','xtick.color':'#8b949e',
    'ytick.color':'#8b949e','text.color':'#c9d1d9','grid.color':'#21262d',
    'grid.linewidth':0.6,'legend.facecolor':'#161b22','legend.edgecolor':'#30363d',
    'legend.labelcolor':'#c9d1d9','font.size':11,'axes.titlesize':13,'axes.titleweight':'bold',
})
C = dict(latent='#58a6ff', observed='#3fb950', censored='#f85149',
         capacity='#d29922', shock='#f85149', trend='#bc8cff',
         seasonal='#3fb950', dow='#79c0ff', noise='#8b949e',
         first='#bc8cff', business='#58a6ff', premium_eco='#3fb950', economy='#79c0ff')

ROOT = os.path.join('..', 'data')
df_fl = pd.read_csv(os.path.join(ROOT, 'flights_latent.csv'), parse_dates=['date'])
df_od = pd.read_csv(os.path.join(ROOT, 'OD_LatentDemand.csv'),  parse_dates=['date'])

print(f"Loaded: {df_fl.date.min().date()} → {df_fl.date.max().date()}")
print(f"  {df_fl.flight_od.nunique()} flights | {df_od.od_pair.nunique()} OD pairs | "
      f"{len(df_fl):,} flight-day rows")

---
## Layer 1 — Market demand

### 1a. μ_OD: the fixed market size

`OD_MARKET_SIZE` in `config.py` defines how many passengers per day want to travel each corridor **at the base year**, independent of how many flights exist.

This is the only place demand is created. Adding a flight does not add demand — it adds capacity to serve existing demand.

In [ ]:
from simulator.config import OD_MARKET_SIZE, ROUTES, AIRCRAFT, AIRPORT_COUNTRY

# Total seat capacity per OD pair (Layer 2 supply — separate from demand)
od_cap = {}
for r in ROUTES:
    dest = AIRPORT_COUNTRY.get(r.dest_airport, '??')
    od = f"{r.market_country}→{dest}"
    od_cap[od] = od_cap.get(od, 0) + AIRCRAFT.get(r.aircraft_type, AIRCRAFT['B789']).total_seats

summary = pd.DataFrame({
    'μ_OD (market size)': OD_MARKET_SIZE,
    'Total seat capacity': od_cap,
}).sort_values('μ_OD (market size)', ascending=False)
summary['Implied avg LF'] = (summary['μ_OD (market size)'] / summary['Total seat capacity']).map('{:.1%}'.format)

print("OD Market Sizes vs Fleet Capacity")
print("=" * 55)
print(summary.to_string())
print()
print("Key: μ_OD does not change when you add/remove a route.")
print("     Only 'Total seat capacity' changes — demand stays fixed.")

### 1b. S(t): the multiplier stack

Every day, five independent multipliers are applied to μ_OD:

| Multiplier | Source | What it models |
|---|---|---|
| `seasonal(t)` | `seasonality.py`, per-country profile | Month-of-year travel peaks (GW, Obon, school hols) |
| `dow(t)` | `seasonality.py` | Day-of-week patterns (Fri peak, Sun trough) |
| `trend(t)` | `country.demand_trend` | Compound annual growth, accelerating early |
| `shock(t)` | `shocks.py`, ShockEngine | Currency crises, pandemics, geopolitical events |
| `noise(t)` | Gaussian, σ=0.08 | Residual daily randomness |

`λ_OD(t) = μ_OD · seasonal · dow · trend · shock · noise`

In [ ]:
# Show the multiplier stack for JP→US over the full horizon
jp = df_od[df_od.od_pair == 'JP→US'].sort_values('date').copy()
mu = OD_MARKET_SIZE['JP→US']

fig, axes = plt.subplots(5, 1, figsize=(15, 12), sharex=True)
fig.suptitle('JP→US — λ_OD(t) multiplier decomposition', fontsize=14, y=0.98)

pairs = [
    ('seasonal_index', 'Seasonal S(month)', C['seasonal']),
    ('dow_factor',     'Day-of-week S(dow)', C['dow']),
    ('trend_factor',   'Trend S(t)', C['trend']),
    ('shock_factor',   'Shock S(t)', C['shock']),
]
for ax, (col, label, color) in zip(axes[:4], pairs):
    ax.plot(jp.date, jp[col], color=color, lw=1.2, alpha=0.85)
    ax.axhline(1.0, color='#8b949e', lw=0.8, ls='--', alpha=0.5)
    ax.set_ylabel(label, fontsize=9)
    ax.grid(True, axis='y')
    if col == 'shock_factor':
        ax.fill_between(jp.date, jp[col], 1.0,
                        where=jp[col] < 0.99, color=C['shock'], alpha=0.25)

# Bottom panel: the resulting λ_OD(t) vs μ_OD
ax5 = axes[4]
ax5.plot(jp.date, jp.expected_demand, color=C['latent'], lw=1.5, label='λ_OD(t) = μ · S(t)')
ax5.axhline(mu, color=C['capacity'], lw=1.5, ls='--', label=f'μ_JP→US = {mu:,} (base)')
ax5.set_ylabel('Passengers/day', fontsize=9)
ax5.legend(loc='upper left', fontsize=9)
ax5.grid(True, axis='y')

axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

---
## Layer 2 — Supply allocation

Once λ_OD(t) is known, it is split across flights **proportionally by seat count**. This is allocation only — no new demand is created here.

```python
flight_share     = aircraft.total_seats / sum(all seats on this OD)
flight_expected  = λ_OD(t) · flight_share
```

A B777 (396 seats) on JP→US gets a larger slice than a B787 (246 seats). If a new flight is added, every existing flight's share shrinks — but the total corridor demand stays at λ_OD(t).

In [ ]:
ROUTE     = 'HND->LAX'
YEAR_SHOW = 2019

data = df_fl[(df_fl.flight_od == ROUTE) & (df_fl.date.dt.year == YEAR_SHOW)].sort_values('date')
cap  = int(data.seats_capacity.iloc[0])

fig, ax = plt.subplots(figsize=(14, 4))

ax.fill_between(data.date, data.observed_seats, data.latent_seats,
                color=C['censored'], alpha=0.3, label='Spillage')
ax.plot(data.date, data.latent_seats,   color=C['latent'],   lw=1.5, label='Latent (oracle)')
ax.plot(data.date, data.observed_seats, color=C['observed'], lw=1.5, label='Observed (realised)')
ax.axhline(cap, color=C['capacity'], lw=1.2, ls='--', label=f'Capacity ({cap})')

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.set_title(f'{ROUTE} — latent vs realised ({YEAR_SHOW})')
ax.set_ylabel('Passengers')
ax.legend(loc='upper left')
ax.grid(True, axis='y')

plt.tight_layout()
plt.show()

---
## Layer 3 — Booking simulation & censoring

`flight_expected` is the Poisson mean for one flight on one departure date. The booking simulation (`simulation.py`) converts it into actual seat counts by simulating the entire **90-day booking window**.

### How the booking window works

For each day-prior-to-departure (DTP from 90 → 1), for each cabin:

```
λ_cabin_dtp = flight_expected · cabin_weight · booking_curve[dtp][cabin]
arrivals     = Poisson(λ_cabin_dtp)
accepted     = min(cumulative_arrivals, cabin_capacity)   # FCFS
```

- `cabin_weight` = that cabin's seats / total aircraft seats
- `booking_curve` is a Beta distribution — business books early, economy books late
- FCFS means: once a cabin is full, further arrivals are **oracle only** (latent but unobservable)

`is_censored = 1` the moment any arrival is rejected.

In [ ]:
# ── Pull one flight's BOH data from flights_latent ────────────────────────────
ROUTE = 'HND->LAX'
flt   = df_fl[df_fl.flight_od == ROUTE].sort_values('date')

boh_cols = sorted([c for c in df_fl.columns if c.startswith('boh_dtp_')],
                  key=lambda x: -int(x.split('_')[-1]))
dtps = [int(c.split('_')[-1]) for c in boh_cols]
cap  = int(flt.seats_capacity.iloc[0])

# Pick a few individual departures to overlay
sample_dates = flt.date.dt.to_period('Q').drop_duplicates().dt.to_timestamp()
sample_rows  = [flt[flt.date >= d].iloc[0] for d in sample_dates if len(flt[flt.date >= d]) > 0]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f'{ROUTE} — BOH build-up from flights_latent.csv  (cap={cap})', fontsize=13)

# Left: individual departure traces
ax = axes[0]
for row in sample_rows:
    boh = [int(row[c]) for c in boh_cols]
    color = C['censored'] if row.is_censored else C['observed']
    ax.plot(dtps, boh, color=color, lw=1.3, alpha=0.7)

# Legend proxies
from matplotlib.lines import Line2D
ax.add_artist(ax.legend(handles=[
    Line2D([0],[0], color=C['censored'], lw=2, label='Censored departure'),
    Line2D([0],[0], color=C['observed'], lw=2, label='Uncensored departure'),
], fontsize=9, loc='upper right'))
ax.axhline(cap, color=C['capacity'], lw=1.5, ls='--', label=f'Capacity ({cap})')
ax.set_xlim(max(dtps), 1)
ax.set_xlabel('Days prior to departure')
ax.set_ylabel('Seats booked (BOH)')
ax.set_title('One departure per quarter — actual BOH trajectories\n(red = hit capacity, green = did not)')
ax.grid(True, axis='y')

# Right: per-cabin observed pax for the same sample departures
ax2 = axes[1]
x = np.arange(len(sample_rows))
bottoms = np.zeros(len(sample_rows))
for cab in ['first', 'business', 'premium_eco', 'economy']:
    vals = np.array([int(r[f'observed_{cab}_pax']) for r in sample_rows])
    ax2.bar(x, vals, bottom=bottoms, color=C[cab], alpha=0.85,
            label=cab.replace('_', ' ').title())
    bottoms += vals

ax2.axhline(cap, color=C['capacity'], lw=1.5, ls='--', label=f'Capacity ({cap})')
ax2.set_xticks(x)
ax2.set_xticklabels([r.date.strftime('%b %Y') for r in sample_rows], rotation=35, ha='right', fontsize=8)
ax2.set_ylabel('Observed passengers (truncated)')
ax2.set_title('Cabin breakdown per departure\n(observed = truncated at capacity)')
ax2.legend(fontsize=8, loc='upper left')
ax2.grid(True, axis='y')

plt.tight_layout()
plt.show()

---
## What the data looks like — oracle vs observed

`oracle_{cabin}_pax` = everyone who tried to book (latent, unobservable in real RM)  
`observed_{cabin}_pax` = everyone who actually got a seat (what an airline sees)  
`is_censored` = 1 when at least one passenger was turned away

The gap between oracle and observed is what unconstraining models try to recover.

In [ ]:
from simulator.booking_curves import (
    generate_booking_curves, cumulative_booking_curve, BookingCurveParams, SEGMENTS
)
import pandas as pd

df_fl2 = pd.read_csv('../data/flights_latent.csv', parse_dates=['date'])
df_fl2['spillage'] = df_fl2.latent_seats - df_fl2.observed_seats
row = df_fl2.nlargest(1, 'spillage').iloc[0]

print(f"Flight: {row.flight_od}  Date: {row.date.date()}")
print(f"Latent: {int(row.latent_seats)}  Observed: {int(row.observed_seats)}  Spillage: {int(row.spillage)}")

oracle_by_cabin = {
    'first':       int(row.oracle_first_pax),
    'business':    int(row.oracle_business_pax),
    'premium_eco': int(row.oracle_premium_eco_pax),
    'economy':     int(row.oracle_economy_pax),
}
cap_by_cabin = {
    'first':       int(row.first_capacity),
    'business':    int(row.business_capacity),
    'premium_eco': int(row.premium_eco_capacity),
    'economy':     int(row.economy_capacity),
}
total_cap = sum(cap_by_cabin.values())

df_curves = generate_booking_curves(BookingCurveParams(booking_window_days=90))
df_cumul  = cumulative_booking_curve(df_curves)

cabin_cumul    = {cab: df_cumul[cab] * oracle_by_cabin[cab] for cab in SEGMENTS}
cabin_observed = {cab: cabin_cumul[cab].clip(upper=cap_by_cabin[cab]) for cab in SEGMENTS}
oracle_total   = sum(cabin_cumul.values())
observed_total = sum(cabin_observed.values())

fig, ax = plt.subplots(figsize=(13, 6))

for cab in SEGMENTS:
    a = SEGMENTS[cab]['alpha']
    b = SEGMENTS[cab]['beta']
    ax.plot(df_cumul.index, cabin_cumul[cab].values,
            color=C[cab], lw=1.8, ls='--',
            label=f'{cab.replace("_"," ").title()}  (α={a}, β={b})')

ax.plot(df_cumul.index, oracle_total.values,
        color='#c9d1d9', lw=2.8,
        label=f'Oracle / True Demand  ({int(row.latent_seats)} pax)')

ax.plot(df_cumul.index, observed_total.values,
        color='#f85149', lw=2.8,
        label=f'Observed / Truncated  ({int(row.observed_seats)} pax)')

ax.fill_between(df_cumul.index, observed_total.values, oracle_total.values,
                color='#f85149', alpha=0.15,
                label=f'Unobserved Demand  ({int(row.spillage)} pax)')

ax.axhline(total_cap, color='#8b949e', lw=1.2, ls=':',
           label=f'Total Capacity  ({total_cap})')

ax.set_xlim(df_cumul.index.min() - 2, 0)
ax.set_xlabel('Days Before Departure')
ax.set_ylabel('Cumulative Passengers')
ax.set_title(f'Segmented Booking Curves — {row.flight_od}  ({row.date.date()})  '
             f'spillage = {int(row.spillage)} pax')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()

---
## Shocks — sudden demand disruption

`ShockEngine` pre-generates three types of events before the simulation runs:

| Type | Who it hits | Example |
|---|---|---|
| **Global** | Every market simultaneously | Pandemic, oil crisis |
| **Country** | One origin market | Currency crash, visa ban |
| **Geopolitical** | Bilateral pair (origin ↔ dest) | Trade war, territorial dispute |

Each shock has a random severity and duration. Effect decays exponentially back to 1.0. The multiplier is cached as a `(n_months × n_countries)` array — one fast lookup per OD per day.

In [ ]:
# Show shock events on JP→US and KR→US — two markets affected differently
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
fig.suptitle('Shock multiplier S(t) — how events hit different markets', fontsize=14)

for ax, pair in zip(axes, ['JP→US', 'KR→US']):
    sub = df_od[df_od.od_pair == pair].sort_values('date')
    mu  = OD_MARKET_SIZE[pair]

    # Shade shock zones
    shock_below = sub.shock_factor < 0.97
    ax.fill_between(sub.date, sub.expected_demand, mu,
                    where=shock_below & (sub.expected_demand < mu),
                    color=C['shock'], alpha=0.35, label='Shock suppression')
    ax.fill_between(sub.date, sub.expected_demand, mu,
                    where=~shock_below & (sub.expected_demand > mu),
                    color=C['trend'], alpha=0.2, label='Trend lift')

    ax.plot(sub.date, sub.expected_demand, color=C['latent'], lw=1.3, alpha=0.9,
            label='λ_OD(t) with all multipliers')
    ax.axhline(mu, color=C['capacity'], lw=1.5, ls='--', alpha=0.7,
               label=f'μ_OD = {mu:,} (baseline)')

    # Annotate deepest shock
    worst_idx = sub.shock_factor.idxmin()
    worst = sub.loc[worst_idx]
    ax.annotate(f"shock={worst.shock_factor:.2f}",
                xy=(worst.date, worst.expected_demand),
                xytext=(worst.date, worst.expected_demand - mu*0.08),
                fontsize=8, color=C['shock'],
                arrowprops=dict(arrowstyle='->', color=C['shock'], lw=1))

    ax.set_ylabel('Pax/day')
    ax.set_title(f'{pair}  (μ_OD fixed — only shock+season+trend move the line)')
    ax.legend(fontsize=8, loc='upper left', ncol=4)
    ax.grid(True, axis='y')

axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

# Shock multiplier distribution
fig2, ax = plt.subplots(figsize=(10, 3))
shock_vals = df_od.groupby(['od_pair','date'])['shock_factor'].first()
ax.hist(shock_vals, bins=80, color=C['shock'], alpha=0.75, edgecolor='none')
ax.axvline(1.0, color='#c9d1d9', lw=1.5, ls='--', label='No shock (=1.0)')
ax.set_xlabel('Shock multiplier value')
ax.set_ylabel('Day-OD observations')
ax.set_title('Distribution of shock multiplier across all OD pairs and days')
ax.legend(); ax.grid(True, axis='y')
pct_shocked = (shock_vals < 0.97).mean()
print(f"Days with shock_factor < 0.97: {pct_shocked:.1%} of all OD-day observations")
plt.tight_layout(); plt.show()

---
## Seasonal patterns

Each country has its own `season_profile` — a 12-element array of monthly demand indices relative to the annual average. These capture: Japanese Golden Week (April–May), summer leisure peaks, Lunar New Year surges, and low-season troughs.

In [ ]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# ── Heatmap: avg seasonal index by market_country × month ─────────────────────
pivot = (
    df_od.groupby(['market_country','month'])['seasonal_index']
    .mean()
    .unstack(level='month')
)
pivot.columns = month_names

fig, axes = plt.subplots(1, 2, figsize=(16, 4),
                          gridspec_kw={'width_ratios': [2.5, 1]})

# Left: heatmap
im = axes[0].imshow(pivot.values, aspect='auto', cmap='RdYlGn',
                     vmin=0.7, vmax=1.4, interpolation='nearest')
axes[0].set_xticks(range(12)); axes[0].set_xticklabels(month_names, fontsize=9)
axes[0].set_yticks(range(len(pivot))); axes[0].set_yticklabels(pivot.index, fontsize=9)
for i in range(len(pivot)):
    for j in range(12):
        v = pivot.values[i, j]
        axes[0].text(j, i, f'{v:.2f}', ha='center', va='center',
                     fontsize=7, color='#0e1117' if 0.85 < v < 1.3 else '#f0f6fc')
axes[0].set_title('Seasonal index by market — monthly average across all years\n(1.0 = baseline, >1 = above-average demand)')
plt.colorbar(im, ax=axes[0], shrink=0.9, label='Seasonal index')

# Right: line plot for top-3 markets
top_markets = df_od.groupby('market_country')['expected_demand'].mean().nlargest(3).index
for mkt in top_markets:
    row = pivot.loc[mkt].values if mkt in pivot.index else None
    if row is not None:
        axes[1].plot(range(12), row, marker='o', ms=4, lw=1.8, label=mkt)
axes[1].set_xticks(range(12)); axes[1].set_xticklabels(month_names, rotation=45, fontsize=8)
axes[1].axhline(1.0, color='#8b949e', ls='--', lw=1, alpha=0.7)
axes[1].set_ylabel('Seasonal index')
axes[1].set_title('Top-3 markets\nseasonality profiles')
axes[1].legend(fontsize=9); axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## Network-wide trends — latent vs observed over 10 years

Across the full simulation horizon, three forces compound: demand grows via `demand_trend`, shocks periodically suppress it, and capacity limits leave a growing fraction of demand unobserved on busy corridors.

In [ ]:
# ── Network-wide monthly aggregates ───────────────────────────────────────────
fl_monthly = (
    df_fl.groupby(df_fl.date.dt.to_period('M'))
    .agg(
        latent_seats=('latent_seats','sum'),
        observed_seats=('observed_seats','sum'),
        censored_flights=('is_censored','sum'),
        total_flights=('is_censored','count'),
        revenue_usd=('revenue_usd','sum'),
    )
    .reset_index()
)
fl_monthly['date'] = fl_monthly['date'].dt.to_timestamp()
fl_monthly['cens_pct'] = fl_monthly['censored_flights'] / fl_monthly['total_flights'] * 100
fl_monthly['spillage'] = fl_monthly['latent_seats'] - fl_monthly['observed_seats']

od_monthly = (
    df_od.groupby(df_od.date.dt.to_period('M'))
    .agg(expected_demand=('expected_demand','sum'))
    .reset_index()
)
od_monthly['date'] = od_monthly['date'].dt.to_timestamp()

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
fig.suptitle('Network-wide simulation output — all OD pairs, all flights', fontsize=14, y=0.99)

# Panel 1: latent vs observed seats
ax = axes[0]
ax.fill_between(fl_monthly.date, fl_monthly.observed_seats/1000, fl_monthly.latent_seats/1000,
                color=C['censored'], alpha=0.35, label='Spillage (censored demand)')
ax.plot(fl_monthly.date, fl_monthly.latent_seats/1000,  color=C['latent'],   lw=1.8,
        label='Oracle seats (true latent)')
ax.plot(fl_monthly.date, fl_monthly.observed_seats/1000, color=C['observed'], lw=1.8,
        label='Observed seats (sold)')
ax.set_ylabel("Seats ('000s)")
ax.set_title('Total monthly seats — oracle vs observed (gap = censored demand that never appeared in the data)')
ax.legend(fontsize=9, loc='upper left'); ax.grid(True, axis='y')

# Panel 2: censoring rate and spillage
ax2 = axes[1]
ax2.plot(fl_monthly.date, fl_monthly.cens_pct, color=C['censored'], lw=1.8,
         label='% flights censored')
ax2.set_ylabel('% flights censored', color=C['censored'])
ax2.tick_params(axis='y', labelcolor=C['censored'])

ax2r = ax2.twinx()
ax2r.fill_between(fl_monthly.date, fl_monthly.spillage/1000, alpha=0.25, color=C['shock'])
ax2r.plot(fl_monthly.date, fl_monthly.spillage/1000, color=C['shock'], lw=1.2,
          label="Spillage ('000s pax)")
ax2r.set_ylabel("Monthly spillage ('000s pax)", color=C['shock'])
ax2r.tick_params(axis='y', labelcolor=C['shock'])
ax2.set_title('Censoring rate and spillage — what unconstraining needs to recover')
ax2.grid(True, axis='y'); ax2.legend(loc='upper left', fontsize=9)

# Panel 3: monthly revenue
ax3 = axes[2]
ax3.bar(fl_monthly.date, fl_monthly.revenue_usd/1e6, width=25,
        color=C['trend'], alpha=0.8, label='Monthly revenue')
ax3.set_ylabel('Revenue (USD M)')
ax3.set_title('Observed revenue — cabin-weighted yield × sold seats (connecting pax discounted)')
ax3.legend(fontsize=9); ax3.grid(True, axis='y')

axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

print(f"Total simulation period:   {df_fl.date.min().date()} → {df_fl.date.max().date()}")
print(f"Total oracle seats:        {fl_monthly.latent_seats.sum():>12,.0f}")
print(f"Total observed seats:      {fl_monthly.observed_seats.sum():>12,.0f}")
print(f"Total spillage:            {fl_monthly.spillage.sum():>12,.0f}  ({fl_monthly.spillage.sum()/fl_monthly.latent_seats.sum():.1%} of latent)")
print(f"Avg censoring rate:        {fl_monthly.cens_pct.mean():.1f}%")
print(f"Total revenue:             ${fl_monthly.revenue_usd.sum()/1e9:.2f}B")

---
## Data schema reference

### `OD_LatentDemand.csv` — one row per (od_pair, date)

| Column | Type | Description |
|---|---|---|
| `date` | date | Departure date |
| `od_pair` | str | E.g. `JP→US` — origin market ISO-2 → dest country ISO-2 |
| `market_country` | str | Origin market ISO-2 |
| `dest_country` | str | Destination country ISO-2 |
| `n_flights` | int | Number of flights serving this OD on this date |
| `total_capacity` | int | Total seats across all flights on this OD |
| `expected_demand` | float | λ_OD(t) — Poisson mean before any capacity constraint |
| `latent_seats` | int | Sum of per-flight oracle Poisson draws (true demand) |
| `yield_economy_usd` | float | Economy yield — driven by demand/baseline ratio |
| `shock_factor` | float | Shock multiplier applied this day (1.0 = no shock) |
| `seasonal_index` | float | Month-of-year multiplier from country season profile |
| `dow_factor` | float | Day-of-week multiplier |
| `trend_factor` | float | Long-run growth multiplier at this point in time |

---

### `flights_latent.csv` — one row per (flight_od, date)

| Column | Type | Description |
|---|---|---|
| `date` | date | Departure date |
| `od_pair` | str | OD corridor this flight belongs to |
| `flight_od` | str | Specific flight, e.g. `HND->LAX` |
| `is_connecting` | int | 1 if this is a 6th-freedom connecting service |
| `aircraft_type` | str | e.g. `B789`, `B77W` |
| `seats_capacity` | int | Total seats — fixed by aircraft type, same every day |
| `distance_km` | int | Great-circle distance |
| `latent_seats` | int | Oracle passengers — true demand regardless of capacity |
| `observed_seats` | int | Passengers who actually got a seat (FCFS) |
| `is_censored` | int | 1 if any cabin turned away at least one passenger |
| `revenue_usd` | float | Cabin-weighted yield × observed pax, connecting discounted |
| `oracle_{cabin}_pax` | int | Per-cabin: first, business, premium_eco, economy |
| `observed_{cabin}_pax` | int | Per-cabin realized bookings |
| `boh_dtp_{N}` | int | Bookings-on-hand N days before departure (N = 330…1) |
| `shock_factor` | float | Inherited from OD-level demand model |
| `seasonal_index` | float | Inherited from OD-level demand model |
| `dow_factor` | float | Inherited from OD-level demand model |
| `trend_factor` | float | Inherited from OD-level demand model |

---

**Key relationships:**
- `latent_seats` ≥ `observed_seats` always; the gap is spillage
- `is_censored = 1` ↔ `latent_seats > observed_seats` (but not the reverse — a partially censored flight still has `is_censored=1`)
- `boh_dtp_1` ≈ `observed_seats` (last snapshot before departure)
- Unconstraining target: given `observed_{cabin}_pax` and `is_censored`, recover `oracle_{cabin}_pax`